<a href="https://colab.research.google.com/github/hyperdbio/Android-inject/blob/master/ex01_YOLO_%EB%B6%84%EB%A5%98%EC%8B%A4%EC%8A%B5(%EC%9D%8C%EC%8B%9D%EC%82%AC%EC%A7%84).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# cpu
# 드라이브
# cd
%cd '/content/drive/MyDrive/YOLO_study'

/content/drive/MyDrive/YOLO_study


## 목표
  - kaggle 에서 제공된 음식 이미지 사진을 활용하여, YOLO 모델의 분류 Task(과제)를 수행해보자
  - 음식 이미지 데이터셋으로 커스텀 YOLO모델을 만들어보자
  - YOLO모델이 이해할 수 있는 데이터의 구조로 변경해보는 과정을 알아보자

In [ ]:
# food: apple pie, donuts, onion_rings, pizza
# 4개의 클래스가 존재

In [3]:
# food.zip 압축 해제
# zipfile > ZipFile(), extractall('해제경로')
# %cd 와 같은 명령을 통해서도 압축 해제가 가능
!unzip ./data/food.zip -d ./data/food/

# ./data/food.zip : 압축 해제 대상 경로 설정
# -d ./data/food/ : 압축 해제 후 저장할 디렉토리 설정

# 현재 이미지 구조
# ./data/food/apple_pie/...jpg
# ./data/food/donuts/...jpg
# ./data/food/onion_rings/...jpg
# ./data/food/pizza/1836888.jpg
# --> kaggle 사이트에서 정리된 폴더 구조 -> yolo모델이 이해할 수 있는 구조가 x

Archive:  ./data/food.zip
   creating: ./data/food/apple_pie/
  inflating: ./data/food/apple_pie/3127422.jpg  
  inflating: ./data/food/apple_pie/2106005.jpg  
  inflating: ./data/food/apple_pie/1847621.jpg  
  inflating: ./data/food/apple_pie/456190.jpg  
  inflating: ./data/food/apple_pie/1461580.jpg  
  inflating: ./data/food/apple_pie/2766725.jpg  
  inflating: ./data/food/apple_pie/222074.jpg  
  inflating: ./data/food/apple_pie/960233.jpg  
  inflating: ./data/food/apple_pie/2439188.jpg  
  inflating: ./data/food/apple_pie/1159801.jpg  
  inflating: ./data/food/apple_pie/2781167.jpg  
  inflating: ./data/food/apple_pie/3335126.jpg  
  inflating: ./data/food/apple_pie/3421349.jpg  
  inflating: ./data/food/apple_pie/1077964.jpg  
  inflating: ./data/food/apple_pie/3748095.jpg  
  inflating: ./data/food/apple_pie/1305678.jpg  
  inflating: ./data/food/apple_pie/1725573.jpg  
  inflating: ./data/food/apple_pie/2704442.jpg  
  inflating: ./data/food/apple_pie/3270291.jpg  
  inflatin

### YOLO 폴더 구조 준비
- 훈련용, 검증용, 평가용으로 분할
- 폴더 구조 가이드라인: https://docs.ultralytics.com/ko/datasets/classify/
- food_for_yolo (전체 폴더 이름)
  - train
    - apple_pie, donuts, .....
  - test
    - apple_pie, donuts, .....
  - val
    - apple_pie, donuts, .....

In [6]:
import os
# 파일 다루기 위한 라이브러리,
# 파일 이름 디렉토리 정보 확인 등 os 관련된 명령을 수행하는 라이브러리

In [12]:
# 각 이미지 파일명 접근
apple_pie_fn = os.listdir('./data/food/apple_pie') #1000
print(len(apple_pie_fn))

donuts_fn = os.listdir('./data/food/donuts') #1000
print(len(donuts_fn))

onion_rings_fn = os.listdir('./data/food/onion_rings') #1000
print(len(onion_rings_fn))

onion_rings_fn = os.listdir('./data/food/onion_rings') #1000
print(len(onion_rings_fn))

# 총 4000장
# train, test, val 폴더로 이동시킬 것
# train - 음식1, 음식2..... 음식4
# 해당 폴더 구조부터 만들고 이동해야함..!

1000
1000
1000
1000


In [21]:
import os
# 폴더생성하기(train, test, val)
# 해당 폴더가 생성되어 있는가?
# os.path.exists('./data/food/pizza') : 있으면 True
# os.path.exists('./data/food_for_yolo') : 없으면 False

if not os.path.exists('./data/food_for_yolo'):
  # 생성해~ : os.mkdir
      os.mkdir('./data/food_for_yolo')

if not os.path.exists('./data/food_for_yolo/train'):
    os.mkdir('./data/food_for_yolo/train')

if not os.path.exists('./data/food_for_yolo/test'):
    os.mkdir('./data/food_for_yolo/test')

if not os.path.exists('./data/food_for_yolo/val'):
    os.mkdir('./data/food_for_yolo/val')



In [37]:
# 하위 폴더 생성하기
# ./data/food_for_yolo/train/apple_pie
# ./data/food_for_yolo/train/donuts
# ./data/food_for_yolo/train/onion_rings
# ./data/food_for_yolo/train/pizza

#food_list = ['apple_pie', 'donuts', 'onion_rings', 'pizza']
food_list = os.listdir('./data/food')
# - 오타 없이 카테고리 이름을 지정하는 명확하고, 신속한 방법
folder_list = ['train', 'test', 'val']

for folder,food in zip(folder_list, food_list):
    for food in food_list:
      if not os.path.exists('./data/food_for_yolo/'+folder+'/'+food):
          os.mkdir('./data/food_for_yolo/'+folder+'/'+food)

In [40]:
# 1000장씩
# train 640
# val 200
# test 160
# gui로 진행하기에는 많은 양(시간 오래 걸림, 딜레이가 발생)

# * graphic user interface : 버튼, 눈에 보이는 것으로 마우스로 클릭해서 처리